# Avance 1.B — Pronóstico de demanda

**CaseStudy: Bebidas del Maipo S.A. — Equipo 4**  
**Objetivo:** estimar la demanda de 9 SKU durante las próximas 13 semanas e informar su incertidumbre para las decisiones de producción e inventario.

Los resultados de calidad de la Parte 1.A se integrarán posteriormente en la planificación de producción.

**Cómo usarlo en Colab:** `Entorno de ejecución → Ejecutar todas`. Las celdas de validación ajustan muchos modelos y pueden tardar varios minutos. Ejecuta en orden; se guardarán tablas y gráficos en `salidas_avance1b/`.



## 0. Diseño del análisis antes de ajustar modelos

**Decisiones que no se cambian mirando la prueba:** unidad = cajas por SKU por semana; horizonte de 13 semanas; estacionalidad aproximada de 52 semanas; desarrollo hasta semana 136; prueba en semanas 137–156; selección por WAPE ponderado en SKU, familia y planta.

**Cuidado:** la venta registrada no es automáticamente la demanda cuando existieron días sin stock. Las correcciones de inventario y errores de unidad son **estimaciones o señales para revisión**, no hechos comprobados en el ERP.



### 0.1. Dependencias reproducibles

Se fijan las versiones de las dependencias. En Colab se instalan solo cuando la versión del entorno es distinta; si instala paquetes, podría solicitar reiniciar la sesión. No se requiere GPU.

In [10]:
# Instalamos solo las dependencias que no coinciden con las versiones requeridas.
import sys
import subprocess
import importlib.metadata as im

VERSIONES = {
    "numpy": "2.3.5",
    "pandas": "2.2.3",
    "scipy": "1.15.3",
    "matplotlib": "3.10.8",
    "statsmodels": "0.14.5",
}

faltan = []
for paquete, version_requerida in VERSIONES.items():
    try:
        version_actual = im.version(paquete)
    except im.PackageNotFoundError:
        version_actual = None

    if version_actual != version_requerida:
        faltan.append(f"{paquete}=={version_requerida}")

if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *faltan])

print("Dependencias preparadas:", VERSIONES)


Dependencias preparadas: {'numpy': '2.3.5', 'pandas': '2.2.3', 'scipy': '1.15.3', 'matplotlib': '3.10.8', 'statsmodels': '0.14.5'}


### 0.2. Importaciones, parámetros y carpetas

- `CFG` contiene las reglas generales: cambiar un parámetro aquí permite rastrear **qué versión** originó los resultados.
- `SAL` reúne las exportaciones; `CRUDOS` conserva las fuentes **sin modificar**, `TAB` guarda los CSV y `FIG` los gráficos.
- La semilla permite reproducir los resultados. No sustituye la validación temporal.

In [11]:
from pathlib import Path
import sys
import os
import json
import random
import hashlib
import warnings
import time
import platform
import shutil

import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from scipy.stats import norm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.stattools import acf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from IPython.display import display, Markdown, FileLink

SEED = 20261014
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

CFG = {
    "semilla": SEED,
    "m": 52,                    # Ciclo estacional anual aproximado en semanas.
    "h": 13,                    # Semanas que pronosticamos de una sola vez.
    "fin_desarrollo": 136,     # Último dato permitido en la selección.
    "origenes_desarrollo": list(range(104, 124)),
    "origenes_prueba": list(range(136, 144)),
    "ventana_proporciones": 52,
    "recuperacion_censura": 1.0,
    "corregir_unidades": True,
    "pesos": {"SKU": 0.60, "Familia": 0.25, "Planta": 0.15},
    "commit_fuentes": "4f2718dc185dd3a5e94243424e386b558a460178",
}

SAL = Path.cwd() / "salidas_avance1b"
CRUDOS = SAL / "datos_crudos"
TAB = SAL / "tablas"
FIG = SAL / "figuras"

for carpeta in [CRUDOS, TAB, FIG]:
    carpeta.mkdir(parents=True, exist_ok=True)

AVISOS = []  # Registro de problemas de ajuste: nunca ocultamos un fallback.
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.18,
})
pd.set_option("display.max_columns", 18)


AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [ ]:
def tabla(df, nombre, mostrar=True, max_filas=18):
    """Guarda el CSV completo; solo limita las filas visibles en pantalla."""
    df.to_csv(TAB / f"{nombre}.csv", index=False, encoding="utf-8")
    if mostrar:
        display(df.head(max_filas).round(4))
        if len(df) > max_filas:
            print(f"{len(df)} filas completas en tablas/{nombre}.csv")
    return df


def figura(fig, nombre):
    """Muestra una figura y también la exporta como PNG."""
    fig.tight_layout()
    fig.savefig(FIG / f"{nombre}.png", dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def guardar_json(obj, nombre):
    """Guarda resultados y parámetros en un archivo JSON."""
    destino = SAL / nombre
    destino.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str),
                       encoding="utf-8")


# Registrar la configuración antes de mirar la prueba.
guardar_json(CFG, "configuracion.json")
print("Los resultados se guardarán en:", SAL)


## 1. Carga y revisión de integridad

**Qué hacemos:** obtenemos únicamente los tres archivos necesarios para este apartado desde una **revisión fija** del repositorio oficial del curso. Comprobamos sus SHA-256 antes de analizarlos.

**Por qué:** una modificación futura de los CSV podría alterar los resultados. Se fija una revisión del repositorio y se verifican los archivos antes de utilizarlos.

**Si falla internet:** descarga el ZIP de respaldo adjunto, súbelo al panel Archivos de Colab y descomprímelo en `/content/`, dejando los CSV en `/content/datos_avance1b/`. La celda buscará ese respaldo y verificará los mismos hashes.

In [ ]:
import urllib.request

# Huellas SHA-256 esperadas para los archivos de entrada.
HASH_ESPERADO = {
    "demanda_semanal.csv": "449ca7aaa5c64e3958e38b11c023ad22588c1e422bf0ee129c8794f8591d9f8e",
    "maestro_sku.csv": "23fa78966c5db09a8cf3c69949c3d38664a3e472fb48e1f16e81b20568d18cbf",
    "metadatos.json": "95e7880a417b8faa603e623624332af4c1b6e70ddad82e95e4be539327ca89f1",
}

# Carpeta opcional con archivos originales, para trabajar sin internet.
RESPALDO_LOCAL = Path.cwd() / "datos_avance1b"

CONEXIONES = []
for nombre_archivo, hash_esperado in HASH_ESPERADO.items():
    url = (
        "https://raw.githubusercontent.com/LuisRojo-Gonzalez/Operations-Management/"
        f"{CFG['commit_fuentes']}/CaseStudy/Datos/{nombre_archivo}"
    )

    try:
        # El enlace queda fijado a un commit, NO a un archivo mutable de main.
        with urllib.request.urlopen(url, timeout=20) as respuesta:
            contenido = respuesta.read()
        origen = "GitHub docente: revisión fija"
    except Exception as error:
        archivo_respaldo = RESPALDO_LOCAL / nombre_archivo
        if not archivo_respaldo.exists():
            raise RuntimeError(
                f"No se pudo descargar {nombre_archivo} y no existe respaldo local. "
                "Sube y descomprime el ZIP de datos en /content/datos_avance1b/."
            ) from error
        contenido = archivo_respaldo.read_bytes()
        origen = "Respaldo local verificado"

    hash_obtenido = hashlib.sha256(contenido).hexdigest()
    if hash_obtenido != hash_esperado:
        raise ValueError(f"El archivo {nombre_archivo} no coincide con la versión auditada.")

    # Se conserva cada archivo sin transformaciones.
    (CRUDOS / nombre_archivo).write_bytes(contenido)
    CONEXIONES.append({
        "archivo": nombre_archivo,
        "origen": origen,
        "sha256": hash_obtenido,
        "url_fuente": url,
    })

tabla(pd.DataFrame(CONEXIONES), "fuentes_datos")


### 1.1. Leer CSV y construir la jerarquía

`parse_dates` transforma el lunes en una fecha real; no hace falta establecerla como índice para que Pandas la grafique correctamente. Un **SKU** es un producto específico y una **familia** agrupa SKU. Trabajaremos en tres niveles: SKU, familia y planta.

In [ ]:
ventas = pd.read_csv(CRUDOS / "demanda_semanal.csv", parse_dates=["fecha_lunes"])
maestro = pd.read_csv(CRUDOS / "maestro_sku.csv")
meta = json.loads((CRUDOS / "metadatos.json").read_text(encoding="utf-8"))

SKUS = sorted(maestro.sku)
FAMILIAS = sorted(maestro.familia.unique())
FAM = maestro.set_index("sku").familia.to_dict()
BOTELLAS = maestro.set_index("sku").botellas_por_caja.to_dict()

NODOS = ["PLANTA"] + ["FAM_" + familia for familia in FAMILIAS] + SKUS
NIVEL = {
    nodo: "SKU" if nodo in SKUS else "Planta" if nodo == "PLANTA" else "Familia"
    for nodo in NODOS
}
calendario = (
    ventas[["semana", "fecha_lunes"]]
    .drop_duplicates()
    .sort_values("semana")
    .set_index("semana")
    .fecha_lunes
)

print("Columnas disponibles:", ventas.columns.tolist())
display(ventas.head())


In [ ]:
# En vez de reparar datos a ciegas, detenemos la ejecución si fallan reglas básicas.
assert len(ventas) == 156 * 9 and len(SKUS) == 9
assert not ventas.duplicated(["sku", "semana"]).any()
assert ventas.groupby("sku").semana.apply(
    lambda semanas: sorted(semanas) == list(range(1, 157))
).all()
assert (calendario.diff().dropna() == pd.Timedelta(days=7)).all()
assert (ventas.fecha_lunes.dt.dayofweek == 0).all()
assert ventas.ventas_cajas.dropna().ge(0).all()
assert ventas.dias_sin_stock.dropna().between(0, 6).all(), (
    "Con 7 días sin stock no es posible dividir por días disponibles."
)
assert ventas.promocion.isin([0, 1]).all()
assert ventas.precio_efectivo_clp.gt(0).all()
assert ventas.sku.isin(SKUS).all()

print(f"Datos válidos: {len(ventas)} filas, {len(SKUS)} SKU y {len(FAMILIAS)} familias.")
tabla(ventas.isna().sum().rename_axis("columna").reset_index(name="faltantes"),
      "integridad_faltantes")


## 2. ¿Qué aspecto tienen las ventas registradas?

**Qué mirar:** tendencia, repeticiones anuales, períodos de cero, promociones y posibles cambios estructurales. El gráfico **solo usa las semanas de desarrollo**; no se selecciona un modelo por lo que ocurra después de la semana 136.

In [ ]:
desarrollo = ventas[ventas.semana <= CFG['fin_desarrollo']].copy()

fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharex=True)

for ax, sku in zip(axes.flat, SKUS):
    g = desarrollo[desarrollo.sku == sku]
    ax.plot(g.fecha_lunes, g.ventas_cajas, lw=1, color='#24496b', label='Venta registrada')
    p = g[g.promocion == 1]
    ax.scatter(p.fecha_lunes, p.ventas_cajas, s=12, color='#d38921', label='Promoción', zorder=3)
    ax.set_title(sku)
    ax.set_ylabel('Cajas/semana')
    ax.tick_params(axis='x', rotation=30)

axes.flat[0].legend(fontsize=8)

fig.suptitle('Ventas crudas del desarrollo: picos, ceros y cambios que deben explicarse', y=1.02)

figura(fig, '01_ventas_crudas_9_sku')


## 3. De ventas a una **estimación** de demanda

Aquí está una decisión importante del caso. Si un SKU vendió 60 cajas en 5 días disponibles, suponemos un ritmo uniforme y estimamos `60 × 7/5 = 84` cajas para toda la semana. Esto **no demuestra** que la demanda real fue 84: depende del supuesto de ventas perdidas sin sustitución o recuperación.

Cada corrección usa exclusivamente el historial conocido **hasta su semana**:

1. Detectamos posibles errores de unidad con comparación contra semanas pasadas; se **marcan como candidatos** que requieren verificación en ERP.
2. Estimamos demanda cuando hubo `dias_sin_stock`, conservando el dato original.
3. Imputamos faltantes usando información anterior; nunca una interpolación centrada que mire el futuro.
4. Marcamos atípicos, pero **no los borramos automáticamente**.

La función devuelve **dos tablas**: datos preparados y bitácora de cambios.

In [ ]:
def intermitencia(y):
    """Clasifica una serie con ADI y CV² (umbral orientativo, no una prueba causal).

    ADI: separación media entre períodos con demanda positiva.
    CV²: variabilidad relativa del tamaño de la demanda positiva.
    """
    serie = np.asarray(y, dtype=float)
    serie = serie[np.isfinite(serie)]
    positivos = serie[serie > 0]

    if len(positivos) == 0:
        return {"ADI": np.inf, "CV2": np.nan,
                "p_ceros": 1.0, "clase": "sin demanda"}

    adi = len(serie) / len(positivos)
    cv2 = (np.std(positivos, ddof=1) / positivos.mean()) ** 2 if len(positivos) > 1 else 0.0

    if adi < 1.32:
        clase = "suave" if cv2 < 0.49 else "errática"
    else:
        clase = "intermitente" if cv2 < 0.49 else "grumosa"

    return {"ADI": adi, "CV2": cv2,
            "p_ceros": np.mean(serie == 0), "clase": clase}


In [12]:
def preparar(historia, recuperacion=1.0, corregir_unidades=True):
    """Prepara únicamente el pasado recibido, sin usar semanas posteriores.

    Parámetros:
        historia: DataFrame de ventas semanales, ya recortado al origen.
        recuperacion: 0=no reponer ventas perdidas, 1=reposición total del ritmo estimado.
        corregir_unidades: si True, examina errores plausibles de botellas vs cajas.

    Devuelve:
        DataFrame con demanda estimada y marcadores, más bitácora de decisiones.
    """
    filas = []
    bitacora = []

    # No se mezclan semanas ni productos al calcular referencias históricas.
    for sku, grupo in historia.groupby("sku", sort=True):
        ventas_previas = []
        demandas_previas = []

        for fila in grupo.sort_values("semana").itertuples():
            venta = float(fila.ventas_cajas)
            dias_sin_stock = float(fila.dias_sin_stock)
            disponible = np.isfinite(venta) and np.isfinite(dias_sin_stock)

            venta_ajustada = venta
            unidad_corregida = False
            imputada = not disponible

            # La mediana usa como máximo las 13 semanas ANTERIORES.
            ventana_previa = np.asarray(ventas_previas[-13:], dtype=float)
            ventana_previa = ventana_previa[np.isfinite(ventana_previa)]
            mediana_previa = (
                float(np.median(ventana_previa)) if len(ventana_previa) else np.nan
            )

            # Paso A. Señalar y corregir un posible error de unidad.
            posible_error = (
                corregir_unidades and disponible and len(ventana_previa) >= 6
                and mediana_previa > 0
                and intermitencia(ventas_previas)["ADI"] < 1.32
                and venta > 4 * mediana_previa
            )
            if posible_error:
                botellas_por_caja = int(BOTELLAS[sku])
                candidato = venta / botellas_por_caja
                entero = abs(candidato - round(candidato)) < 1e-8
                rango_plausible = 0.45 * mediana_previa <= candidato <= 2.5 * mediana_previa

                if entero and rango_plausible:
                    venta_ajustada = candidato
                    unidad_corregida = True
                    bitacora.append({
                        "sku": sku, "semana": fila.semana,
                        "tipo": "posible_error_unidad", "antes": venta,
                        "despues": venta_ajustada, "referencia_previa": mediana_previa,
                        "regla": f"división por {botellas_por_caja} botellas/caja; confirmar ERP",
                    })

            # Paso B. Demanda estimada: compensar días sin stock.
            if disponible:
                factor = 1 + recuperacion * (7 / (7 - dias_sin_stock) - 1)
                demanda_estimada = venta_ajustada * factor
                if dias_sin_stock > 0:
                    bitacora.append({
                        "sku": sku, "semana": fila.semana, "tipo": "censura",
                        "antes": venta_ajustada, "despues": demanda_estimada,
                        "referencia_previa": mediana_previa,
                        "regla": "ritmo diario uniforme; estimación, no observación",
                    })
            else:
                # Paso C. Faltante: tomar t-52 si existe, si no mediana del pasado.
                posicion_anual = len(demandas_previas) - CFG["m"]
                if posicion_anual >= 0:
                    demanda_estimada = demandas_previas[posicion_anual]
                    regla = "demanda estimada t-52"
                elif demandas_previas:
                    demanda_estimada = float(np.median(demandas_previas[-13:]))
                    regla = "mediana de pasado"
                else:
                    raise ValueError(f"{sku}: faltante inicial sin información para imputar.")

                bitacora.append({
                    "sku": sku, "semana": fila.semana, "tipo": "faltante",
                    "antes": None, "despues": demanda_estimada,
                    "referencia_previa": None, "regla": regla,
                })

            # Paso D. Una alerta estadística no equivale a borrar un valor.
            atipico = False
            if len(ventana_previa) >= 6 and mediana_previa > 0 and not unidad_corregida and disponible:
                mad = float(np.median(abs(ventana_previa - mediana_previa)))
                umbral = max(6 * 1.4826 * mad, 0.75 * mediana_previa)
                atipico = abs(venta_ajustada - mediana_previa) > umbral

                if atipico:
                    bitacora.append({
                        "sku": sku, "semana": fila.semana, "tipo": "atipico_conservado",
                        "antes": venta, "despues": venta_ajustada,
                        "referencia_previa": mediana_previa,
                        "regla": f"bandera robusta; promoción={fila.promocion}; no borrar sin evidencia",
                    })

            # Resultado de la semana: observación cruda + ajustes + trazabilidad.
            filas.append({
                "sku": sku, "semana": fila.semana, "fecha_lunes": fila.fecha_lunes,
                "ventas_originales": venta, "ventas_corregidas": venta_ajustada,
                "demanda_estimada": demanda_estimada,
                "dias_sin_stock": dias_sin_stock, "promocion": fila.promocion,
                "precio_lista_clp": fila.precio_lista_clp,
                "precio_efectivo_clp": fila.precio_efectivo_clp,
                "imputada": imputada, "unidad_corregida": unidad_corregida,
                "atipico": atipico, "objetivo_evaluable": disponible,
            })

            # Actualizamos el historial SOLO DESPUÉS de procesar esta semana.
            ventas_previas.append(venta_ajustada if np.isfinite(venta_ajustada) else np.nan)
            demandas_previas.append(float(demanda_estimada))

    return pd.DataFrame(filas), pd.DataFrame(bitacora)


In [13]:
desarrollo = ventas[ventas.semana <= CFG["fin_desarrollo"]].copy()
lim_dev, bit_dev = preparar(desarrollo)

tabla(bit_dev, "02_bitacora_desarrollo")
tabla(bit_dev[bit_dev.tipo.isin(["posible_error_unidad", "faltante"])],
      "02_anomalias_prioritarias_desarrollo")

# La bitácora indica cuánto se cambió y por qué. No reemplaza revisar el ERP.
print("Registros originales del desarrollo:", len(desarrollo))
print("Registros preparados:", len(lim_dev))


NameError: name 'ventas' is not defined

## 4. Diagnóstico: ¿qué estructura necesita cada serie?

No hay que imponer un único modelo por tener nueve SKU. Calculamos intermitencia, autocorrelación en rezago 52, tendencia descriptiva, promociones y **candidatos** a quiebre. Distingue siempre una correlación/promoción de un efecto causal comprobado.

In [ ]:
def quiebre_candidato(g):
    """Señala candidatos a quiebre comparando razones anuales sin imputaciones.

    Es un indicador exploratorio robusto, NO una confirmación del cambio
    estructural ni una prueba formal de significancia estadística.
    """
    grupo = g.sort_values("semana").set_index("semana")
    demanda = grupo.demanda_estimada
    razon_anual = demanda / demanda.shift(52).replace(0, np.nan)

    valido = (
        (~grupo.imputada)
        & (~grupo.imputada.shift(52, fill_value=True))
        & grupo.promocion.eq(0)
        & grupo.promocion.shift(52).eq(0)
    )
    razones = razon_anual[valido].replace([np.inf, -np.inf], np.nan).dropna()

    if len(razones) < 32:
        return {"semana_quiebre": np.nan, "salto_relativo": np.nan,
                "reduccion_sse": np.nan}

    valores = razones.to_numpy()
    sse_original = np.sum((valores - np.median(valores)) ** 2)
    opciones = []

    for posicion in range(12, len(razones) - 11):
        antes = valores[:posicion]
        despues = valores[posicion:]
        if np.median(antes) <= 0:
            continue

        salto = np.median(despues) / np.median(antes) - 1
        sse = (
            np.sum((antes - np.median(antes)) ** 2)
            + np.sum((despues - np.median(despues)) ** 2)
        )
        opciones.append((sse, posicion, salto))

    sse, posicion, salto = min(opciones)
    reduccion = 1 - sse / sse_original if sse_original > 0 else 0
    detectado = abs(salto) >= 0.25 and reduccion >= 0.20

    return {
        "semana_quiebre": int(razones.index[posicion]) if detectado else np.nan,
        "salto_relativo": salto,
        "reduccion_sse": reduccion,
    }


In [ ]:
diagnosticos = []
promociones = []

for sku, grupo in lim_dev.groupby("sku"):
    grupo = grupo.sort_values("semana")
    demanda = grupo.demanda_estimada.to_numpy()

    # La autocorrelación a 52 semanas aproxima la repetición anual.
    autocorr_52 = acf(demanda, nlags=52, fft=True)[52] if np.std(demanda) > 0 else np.nan
    pendiente = np.polyfit(grupo.semana, demanda, 1)[0]

    diagnosticos.append({
        "sku": sku, "familia": FAM[sku],
        **intermitencia(demanda),
        "pendiente_cajas_semana": pendiente,
        "acf_lag52": autocorr_52,
        "faltantes": int(grupo.imputada.sum()),
        "semanas_censuradas": int((grupo.dias_sin_stock > 0).sum()),
        "posibles_errores_unidad": int(grupo.unidad_corregida.sum()),
        "atipicos_conservados": int(grupo.atipico.sum()),
        "promociones": int(grupo.promocion.sum()),
        **quiebre_candidato(grupo),
    })

    # Comparación descriptiva con la demanda del mismo período del año previo.
    por_semana = grupo.set_index("semana")
    razon_anual = (
        por_semana.demanda_estimada
        / por_semana.demanda_estimada.shift(52).replace(0, np.nan)
    )
    razon_promo = razon_anual[por_semana.promocion.eq(1)]
    razon_sin_promo = razon_anual[por_semana.promocion.eq(0)]
    razon_promo = razon_promo.replace([np.inf, -np.inf], np.nan).dropna()
    razon_sin_promo = razon_sin_promo.replace([np.inf, -np.inf], np.nan).dropna()

    descuento = (
        1 - por_semana.loc[por_semana.promocion.eq(1), "precio_efectivo_clp"]
        / por_semana.loc[por_semana.promocion.eq(1), "precio_lista_clp"]
    ).mean()

    promociones.append({
        "sku": sku,
        "n_promo": len(razon_promo),
        "n_sin_promo": len(razon_sin_promo),
        "mediana_ratio_promo": razon_promo.median(),
        "mediana_ratio_sin_promo": razon_sin_promo.median(),
        "descuento_medio_promo": descuento,
    })

diagnostico = tabla(pd.DataFrame(diagnosticos), "03_diagnostico_9_sku")
tabla(pd.DataFrame(promociones), "03_promociones_asociacion_no_causal")


In [ ]:
# Ventas originales frente a demanda corregida: interpretar las diferencias.
fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharex=True)

for ax, sku in zip(axes.flat, SKUS):
    g = lim_dev[lim_dev.sku == sku]
    ax.plot(g.fecha_lunes, g.ventas_corregidas, label='Venta con unidad revisada', alpha=0.65, lw=1)
    ax.plot(g.fecha_lunes, g.demanda_estimada, label='Demanda estimada', lw=1, color='#b04c38')
    f = g[g.imputada]
    ax.scatter(f.fecha_lunes, f.demanda_estimada, marker='x', color='black', s=35, label='Imputada')
    ax.set_title(sku)
    ax.set_ylabel('Cajas/semana')
    ax.tick_params(axis='x', rotation=30)

axes.flat[0].legend(fontsize=7)

figura(fig, '03_ventas_y_demanda_estimada')


### 4.1. ¿Las familias compensan sus estacionalidades?

Se comparan perfiles de **dos ciclos anuales completos** del desarrollo, normalizando cada familia a media 1. El suavizado centrado se usa **solo para el gráfico diagnóstico**, nunca para entrenar ni pronosticar: así evitamos leakage.

In [ ]:
# Dos ciclos completos; un tercer año parcial podría sesgar el perfil.
anual = lim_dev[lim_dev.semana <= 104].copy()
anual = anual.assign(
    familia=anual.sku.map(FAM),
    fase=(anual.semana - 1) % 52,
)

# El índice normalizado permite comparar la forma de las 3 familias.
perfil = anual.groupby(["familia", "fase"]).demanda_estimada.mean().unstack("familia")
perfil = perfil / perfil.mean()
correlacion = perfil.corr()
tabla(
    correlacion.rename_axis("familia").reset_index(),
    "03_correlacion_perfiles_familia",
)

volumen_familia = anual.groupby("familia").demanda_estimada.sum()
pesos_fam = volumen_familia / volumen_familia.sum()
indice_total = perfil.mul(pesos_fam, axis=1).sum(axis=1)

# Contraste: amplitud de total frente a suma de amplitudes de familias.
sd_sin_cancelacion = sum(pesos_fam[f] * perfil[f].std() for f in FAMILIAS)
cancelacion = 1 - indice_total.std() / sd_sin_cancelacion

tabla(pd.DataFrame([{
    "ciclos_completos": 2,
    "sd_perfil_total": indice_total.std(),
    "cota_sd_correlacion_positiva_perfecta": sd_sin_cancelacion,
    "reduccion_relativa_amplitud": cancelacion,
}]), "03_evidencia_cancelacion_estacional")

fig, ax = plt.subplots(figsize=(11, 4))
for familia in FAMILIAS:
    # Este suavizado centrado SOLO aparece en un gráfico descriptivo.
    perfil_suave = perfil[familia].rolling(5, center=True, min_periods=1).mean()
    ax.plot(perfil.index + 1, perfil_suave, label=familia)
ax.set(
    xlabel="Semana del ciclo desde 2 de octubre (no semana ISO)",
    ylabel="Índice: media de familia = 1",
    title="Fases familiares: agregar puede cancelar parte de la señal",
)
ax.legend()
figura(fig, "03_fases_estacionales_familias")
print("El suavizado centrado SOLO ilustra el perfil; no se usa para pronosticar.")


## 5. Modelos candidatos para pronosticar 13 semanas

**Referencias mínimas:** ingenuo, ingenuo estacional y versión estacional ajustada por nivel. **Alternativas:** ETS con tendencia/estacionalidad, SARIMA estacional y SBA para series intermitentes.

Todos reciben una **historia conocida**, el horizonte `h` y un identificador de modelo. No reciben la semana que se quiere pronosticar.

Importante: **AutoARIMA no se utiliza como selección automática definitiva**; las especificaciones se declararon antes de la prueba. Si un ajuste fracasa, la función registra el problema y utiliza el ingenuo estacional. El informe puede saber cuántas veces sucedió.

In [ ]:
MODELOS = [
    "Ingenuo",
    "Estacional",
    "Estacional_nivel",
    "ETS_AAA",
    "ETS_AdAA",
    "SARIMA_100_010",
    "ETS_SBA",
]

print("Candidatos predefinidos:", MODELOS)


### 5.1. SBA: alternativa cuando aparecen muchos ceros

La corrección de Syntetos–Boylan ajusta por separado **tamaño de demanda positiva** y **tiempo entre ocurrencias**. La usamos únicamente cuando ADI identifica intermitencia; si no, `ETS_SBA` utiliza la especificación ETS predefinida.

In [ ]:
def sba(y, h, alpha=0.1):
    """Demanda intermitente: tamaño esperado corregido / intervalo esperado."""
    tamano = None
    intervalo_suavizado = None
    intervalo_desde_pedido = 1

    for valor in y:
        if valor > 0:
            if tamano is None:
                tamano = float(valor)
                intervalo_suavizado = float(intervalo_desde_pedido)
            else:
                tamano = alpha * valor + (1 - alpha) * tamano
                intervalo_suavizado = (
                    alpha * intervalo_desde_pedido
                    + (1 - alpha) * intervalo_suavizado
                )
            intervalo_desde_pedido = 1
        else:
            intervalo_desde_pedido += 1

    demanda_media = 0 if tamano is None else (1 - alpha / 2) * tamano / intervalo_suavizado
    return np.repeat(demanda_media, h)


### 5.2. Reglas estacionales, ETS y SARIMA

Separamos cada cálculo del control de errores. Esto permite localizar claramente **qué ecuación ejecuta el modelo** y cómo se selecciona y evalúa cada método. En ETS se escala por la media **del entrenamiento** para estabilidad numérica y después se devuelve la magnitud en cajas. SARIMA utiliza diferencia estacional de orden 1 con período 52, y reconstruye el pronóstico en unidades originales.

In [ ]:
def estacional_nivel(y, h, g, pronostico_estacional):
    """Escala el ingenuo estacional por una razón anual reciente y robusta."""
    razones = y[52:] / np.where(y[:-52] > 0, y[:-52], np.nan)

    if g is not None:
        # Comparar solo pares históricos confiables para ajustar el nivel.
        no_imputado = (
            ~g.imputada.to_numpy()[52:]
            & ~g.imputada.to_numpy()[:-52]
        )
        sin_promocion_actual = g.promocion.to_numpy()[52:] == 0
        razones = np.where(no_imputado & sin_promocion_actual, razones, np.nan)

    ultimas_razones = razones[-13:]
    ultimas_razones = ultimas_razones[np.isfinite(ultimas_razones)]
    cambio_nivel = np.median(ultimas_razones) if len(ultimas_razones) >= 4 else 1.0
    return pronostico_estacional * cambio_nivel


def pronostico_ets(y, h, modelo):
    """Suavizamiento exponencial con estacionalidad aditiva (ETS)."""
    escala = max(y.mean(), 1.0)
    especificacion = ExponentialSmoothing(
        y / escala,
        trend="add",
        damped_trend=modelo != "ETS_AAA",
        seasonal="add",
        seasonal_periods=52,
        initialization_method="estimated",
    )
    ajuste = especificacion.fit(optimized=True, use_brute=False)

    if not ajuste.mle_retvals.get("success", True):
        # Segundo intento de la MISMA especificación, no selección por prueba.
        ajuste = especificacion.fit(
            optimized=True, use_brute=False, method="L-BFGS-B",
            minimize_kwargs={"options": {"maxiter": 2000}},
        )
    if not ajuste.mle_retvals.get("success", True):
        raise RuntimeError("ETS no convergió")

    return np.asarray(ajuste.forecast(h)) * escala


def pronostico_sarima(y, h, pronostico_estacional):
    """SARIMA(1,0,0)(0,1,0)[52] con constante; devuelve unidades originales."""
    escala = max(y.mean(), 1.0)
    ajuste = SARIMAX(
        y / escala,
        order=(1, 0, 0),
        seasonal_order=(0, 1, 0, 52),
        trend="c",
        simple_differencing=True,
        enforce_stationarity=True,
        enforce_invertibility=True,
    ).fit(disp=False, maxiter=300)

    if not ajuste.mle_retvals.get("converged", False):
        raise RuntimeError("SARIMA no convergió")
    assert h <= 52, "Para h > 52 debe invertirse la diferencia recursivamente."

    return np.asarray(ajuste.forecast(h)) * escala + pronostico_estacional


In [ ]:
def pronosticar(y, h, modelo, origen, nodo, g=None):
    """Selecciona un método, verifica su salida y registra avisos/fallbacks.

    El fallback al ingenuo estacional se contabiliza posteriormente para
    no declarar como ganador a un modelo que casi nunca logró ajustarse.
    """
    y = np.asarray(y, dtype=float)
    assert np.isfinite(y).all() and len(y) >= 104
    pronostico_estacional = y[-52:][np.arange(h) % 52]

    modelo_usado = modelo
    mensajes = []
    try:
        with warnings.catch_warnings(record=True) as capturadas:
            warnings.simplefilter("always")

            if modelo == "Ingenuo":
                pred = np.repeat(y[-1], h)
            elif modelo == "Estacional":
                pred = pronostico_estacional.copy()
            elif modelo == "Estacional_nivel":
                pred = estacional_nivel(y, h, g, pronostico_estacional)
            elif modelo == "ETS_SBA" and intermitencia(y)["ADI"] >= 1.32:
                pred = sba(y, h)
                modelo_usado = "SBA_alpha_0.1"
            elif modelo in ["ETS_AAA", "ETS_AdAA", "ETS_SBA"]:
                pred = pronostico_ets(y, h, modelo)
            elif modelo == "SARIMA_100_010":
                pred = pronostico_sarima(y, h, pronostico_estacional)
            else:
                raise ValueError(modelo)

            mensajes = [str(aviso.message)[:200] for aviso in capturadas]

        if len(pred) != h or not np.isfinite(pred).all():
            raise ValueError("Pronóstico no finito o largo incorrecto")

    except Exception as error:
        pred = pronostico_estacional.copy()
        modelo_usado = "RETORNO_ESTACIONAL"
        mensajes.append(f"{type(error).__name__}: {error}")

    cantidad_negativos = int((pred < 0).sum())
    pred = np.maximum(pred, 0)

    AVISOS.append({
        "origen": origen, "nodo": nodo,
        "candidato": modelo, "usado": modelo_usado,
        "recortes_negativos": cantidad_negativos,
        "aviso": " | ".join(mensajes),
    })
    return pred


## 6. Jerarquía: SKU → familia → planta

Primero construimos las sumas históricas. Luego aplicamos **tres estrategias** de reconciliación: `BU` (*bottom-up*: pronosticar SKU y sumar); `TD` (*top-down*: pronosticar planta y repartir por volumen reciente); `MO` (pronosticar familias y repartir dentro de ellas). **La repartición usa solo el entrenamiento de cada origen.**

In [ ]:
def jerarquia(datos):
    """Devuelve las series de demanda de SKU, familias y planta por semana."""
    sku_semanal = datos.pivot(
        index="semana", columns="sku", values="demanda_estimada"
    ).reindex(columns=SKUS)

    agregados = pd.DataFrame(index=sku_semanal.index)
    agregados["PLANTA"] = sku_semanal.sum(axis=1)
    for familia in FAMILIAS:
        productos = [sku for sku in SKUS if FAM[sku] == familia]
        agregados["FAM_" + familia] = sku_semanal[productos].sum(axis=1)

    return pd.concat([agregados, sku_semanal], axis=1)[NODOS]


def mascara_jerarquia(datos):
    """Un agregado es evaluable si todas sus observaciones SKU lo son."""
    mascara_sku = datos.pivot(
        index="semana", columns="sku", values="objetivo_evaluable"
    ).reindex(columns=SKUS)

    agregados = pd.DataFrame(index=mascara_sku.index)
    agregados["PLANTA"] = mascara_sku.all(axis=1)
    for familia in FAMILIAS:
        productos = [sku for sku in SKUS if FAM[sku] == familia]
        agregados["FAM_" + familia] = mascara_sku[productos].all(axis=1)

    return pd.concat([agregados, mascara_sku], axis=1)[NODOS]


In [ ]:
def reconciliar(pronosticos_base, historia, estrategia):
    """Convierte pronósticos base en una jerarquía coherente.

    BU: pronosticar cada SKU; TD: repartir pronóstico planta;
    MO: repartir los pronósticos de las familias.
    """
    reciente = historia[SKUS].tail(CFG["ventana_proporciones"])

    if estrategia == "BU":
        hojas = pronosticos_base[SKUS].copy()

    elif estrategia == "TD":
        proporcion_sku = reciente.sum() / reciente.sum().sum()
        assert np.isclose(proporcion_sku.sum(), 1)
        hojas = pd.DataFrame({
            sku: pronosticos_base.PLANTA * proporcion_sku[sku]
            for sku in SKUS
        })

    elif estrategia == "MO":
        hojas = pd.DataFrame(index=pronosticos_base.index)
        for familia in FAMILIAS:
            productos = [sku for sku in SKUS if FAM[sku] == familia]
            participacion = reciente[productos].sum()
            if participacion.sum() > 0:
                participacion = participacion / participacion.sum()
            else:
                participacion = pd.Series(1 / len(productos), index=productos)

            for sku in productos:
                hojas[sku] = pronosticos_base["FAM_" + familia] * participacion[sku]
    else:
        raise ValueError(estrategia)

    # Recalcular los niveles altos garantiza coherencia matemática.
    totales = pd.DataFrame(index=pronosticos_base.index)
    totales["PLANTA"] = hojas.sum(axis=1)
    for familia in FAMILIAS:
        productos = [sku for sku in SKUS if FAM[sku] == familia]
        totales["FAM_" + familia] = hojas[productos].sum(axis=1)
    reconciliado = pd.concat([totales, hojas], axis=1)[NODOS]

    assert np.allclose(reconciliado.PLANTA, reconciliado[SKUS].sum(axis=1))
    for familia in FAMILIAS:
        productos = [sku for sku in SKUS if FAM[sku] == familia]
        assert np.allclose(
            reconciliado["FAM_" + familia], reconciliado[productos].sum(axis=1)
        )
    return reconciliado


In [ ]:
def predecir_origen(crudos, corte, modelos, estrategias,
                   recuperacion=1, corregir_unidades=True):
    """Genera h pronósticos a partir de la historia conocida hasta 'corte'."""
    historia_visible = crudos[crudos.semana <= corte].copy()
    assert historia_visible.semana.max() == corte

    preparados, bitacora = preparar(
        historia_visible, recuperacion, corregir_unidades
    )
    series = jerarquia(preparados)
    predicciones = {}

    for modelo in modelos:
        base = pd.DataFrame(index=np.arange(1, CFG["h"] + 1))
        for nodo in NODOS:
            detalle_sku = (
                preparados[preparados.sku == nodo].sort_values("semana")
                if nodo in SKUS else None
            )
            base[nodo] = pronosticar(
                series[nodo], CFG["h"], modelo, corte, nodo, detalle_sku
            )

        for estrategia in estrategias:
            predicciones[(modelo, estrategia)] = reconciliar(base, series, estrategia)

    return predicciones, series


## 7. Validación temporal: ¿qué modelo funciona antes de ver el futuro?

**Origen rodante:** para cada origen, la función corta la historia, reestima el método, genera **13 pronósticos de una vez** y recién entonces consulta sus valores reales. Los orígenes se superponen: eso da múltiples decisiones de origen, **no observaciones independientes**.

**Métricas:** WAPE mide error absoluto relativo al volumen, MASE lo compara con una escala ingenua estacional, y el sesgo detecta sobre- o subestimación. Solo se califican objetivos evaluables. El resultado de cada modelo/estrategia se exporta para auditoría.

In [ ]:
def evaluar_ventanas(crudos, origenes, modelos, estrategias, etiqueta):
    """Pronostica desde varios orígenes y evalúa solamente semanas posteriores."""
    filas = []
    auditoria = []

    for corte in origenes:
        inicio = time.perf_counter()
        predicciones, entrenamiento = predecir_origen(
            crudos, corte, modelos, estrategias
        )

        # Las semanas objetivo se recuperan DESPUÉS de generar el pronóstico.
        datos_hasta_objetivo = crudos[crudos.semana <= corte + CFG["h"]]
        datos_objetivo, _ = preparar(datos_hasta_objetivo)
        real = jerarquia(datos_objetivo).loc[corte + 1: corte + CFG["h"]]
        evaluable = mascara_jerarquia(datos_objetivo).loc[real.index]

        # Escala MASE: naive estacional calculado SOLO con entrenamiento.
        escala = (entrenamiento - entrenamiento.shift(52)).abs().iloc[52:].mean()
        assert entrenamiento.index.max() == corte
        assert real.index.min() > corte

        for (modelo, estrategia), pronostico in predicciones.items():
            for horizonte in range(1, CFG["h"] + 1):
                semana_objetivo = corte + horizonte
                for nodo in NODOS:
                    valor_real = float(real.loc[semana_objetivo, nodo])
                    estimacion = float(pronostico.loc[horizonte, nodo])

                    # Separa la evaluación de ventas directamente observadas.
                    es_observado = False
                    if nodo in SKUS:
                        registro = datos_objetivo[
                            (datos_objetivo.sku == nodo)
                            & (datos_objetivo.semana == semana_objetivo)
                        ].iloc[0]
                        es_observado = bool(
                            registro.objetivo_evaluable
                            and registro.dias_sin_stock == 0
                            and not registro.unidad_corregida
                        )

                    filas.append({
                        "etapa": etiqueta,
                        "origen": corte,
                        "semana": semana_objetivo,
                        "h": horizonte,
                        "modelo": modelo,
                        "estrategia": estrategia,
                        "nodo": nodo,
                        "nivel": NIVEL[nodo],
                        "y": valor_real,
                        "yhat": estimacion,
                        "e": valor_real - estimacion,
                        "escala_mase": float(escala[nodo]),
                        "evaluable": bool(evaluable.loc[semana_objetivo, nodo]),
                        "sin_censura_ni_unidad": es_observado,
                    })

        duracion = time.perf_counter() - inicio
        auditoria.append({
            "etapa": etiqueta, "origen": corte,
            "max_ajuste": int(entrenamiento.index.max()),
            "min_objetivo": int(real.index.min()),
            "max_objetivo": int(real.index.max()),
            "segundos": duracion,
        })
        print(f"{etiqueta}: origen {corte}, objetivos "
              f"{corte+1}–{corte+CFG['h']}, {duracion:.1f} s")

    return pd.DataFrame(filas), pd.DataFrame(auditoria)


In [14]:
def resumir(ev, grupos):
    """Resumen comparable de errores usando solo pares considerados evaluables."""
    filas = []
    for clave, grupo in ev[ev.evaluable].groupby(grupos, sort=False):
        if not isinstance(clave, tuple):
            clave = (clave,)

        volumen = grupo.y.abs().sum()
        escala = grupo.escala_mase.replace(0, np.nan)
        error = grupo.e

        filas.append({
            **dict(zip(grupos, clave)),
            "n_pares": len(grupo),
            "n_origenes": grupo.origen.nunique(),
            "WAPE_pct": 100 * error.abs().sum() / volumen if volumen else np.nan,
            "MASE": (error.abs() / escala).mean(),
            "sesgo_cajas": error.mean(),
            "sesgo_pct": 100 * error.sum() / volumen if volumen else np.nan,
            "MAE_cajas": error.abs().mean(),
        })
    return pd.DataFrame(filas)


### 7.1. Evaluación de desarrollo: 20 orígenes × 7 modelos × 3 estrategias

Este es el bloque **más costoso en tiempo**, porque reestima modelos ETS/SARIMA. La evaluación permite seleccionar modelos sin utilizar la prueba externa.

In [15]:
# Esta celda hace numerosos ajustes ETS y SARIMA; puede tardar varios minutos.
cv, audit_dev = evaluar_ventanas(
    ventas, CFG["origenes_desarrollo"], MODELOS,
    ["BU", "TD", "MO"], "desarrollo",
)

tabla(cv, "05_predicciones_desarrollo", mostrar=False)
tabla(audit_dev, "05_auditoria_ventanas_desarrollo")
metricas_dev = tabla(
    resumir(cv, ["modelo", "estrategia", "nivel"]),
    "05_comparacion_jerarquica_por_nivel", max_filas=30,
)
por_h = tabla(
    resumir(cv, ["modelo", "estrategia", "nivel", "h"]),
    "05_metricas_por_horizonte_desarrollo", mostrar=False,
)


NameError: name 'evaluar_ventanas' is not defined

### 7.2. Comparación a igualdad de estrategia

Primero mostramos los modelos con `BU`, para que una diferencia de reconciliación no parezca una diferencia del método de pronóstico. Revisamos la evolución de WAPE conforme aumenta el horizonte.

In [ ]:
# BU se mantiene fija para comparar modelos antes de comparar estrategias.
comparacion = metricas_dev[
    (metricas_dev.estrategia == "BU") & (metricas_dev.nivel == "SKU")
].sort_values("WAPE_pct")
tabla(comparacion, "05_modelos_comparables_BU")

fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, nivel in zip(ejes, ["SKU", "Familia", "Planta"]):
    for modelo in MODELOS:
        grupo = por_h[
            (por_h.estrategia == "BU")
            & (por_h.nivel == nivel)
            & (por_h.modelo == modelo)
        ]
        ax.plot(grupo.h, grupo.WAPE_pct, marker=".", label=modelo)
    ax.set(
        title=nivel,
        xlabel="Horizonte h (semanas)",
        ylabel="WAPE (%)",
        xticks=[1, 4, 7, 10, 13],
    )
ejes[2].legend(fontsize=7, bbox_to_anchor=(1.02, 1))
figura(fig, "05_error_por_horizonte_modelos_BU")

# También preservamos el detalle por nodo (no solo promedios de 9 SKU).
tabla(resumir(cv, ["modelo", "estrategia", "nodo", "nivel"]),
      "05_metricas_por_nodo_desarrollo", mostrar=False)


## 8. Selección cerrada, sin utilizar la prueba

El puntaje responde a una **decisión explícita del equipo**, no a una fórmula universal del curso: 60 % WAPE de SKU, 25 % familia, 15 % planta. Una receta no es elegible si su ajuste retorna al ingenuo estacional en más de 5 % de intentos. Se guardan elección y firma SHA-256 **antes** de evaluar el futuro.

In [ ]:
ranking = metricas_dev.pivot(
    index=["modelo", "estrategia"], columns="nivel", values="WAPE_pct"
).reset_index()

# Estos pesos son una decisión operacional declarada por el equipo.
ranking["puntaje"] = sum(
    CFG["pesos"][nivel] * ranking[nivel] for nivel in CFG["pesos"]
)

# Un candidato que casi siempre falla no debe ganar bajo un nombre engañoso.
salud_dev = pd.DataFrame(AVISOS).assign(
    retorno=lambda df: df.usado.eq("RETORNO_ESTACIONAL")
)
tasa_retorno = salud_dev.groupby("candidato").retorno.mean()
ranking["tasa_retorno"] = ranking.modelo.map(tasa_retorno)
ranking["elegible"] = ranking.tasa_retorno.le(0.05)

# Desempate predefinido, sin mirar la prueba.
ranking["orden_modelo"] = ranking.modelo.map(
    {modelo: posicion for posicion, modelo in enumerate(MODELOS)}
)
ranking = ranking.sort_values(
    ["elegible", "puntaje", "orden_modelo", "estrategia"],
    ascending=[False, True, True, True],
).reset_index(drop=True)
tabla(ranking, "06_ranking_seleccion", max_filas=21)

assert ranking.iloc[0].elegible, "Ningún candidato cumple la salud mínima de ajuste."
MODELO_ELEGIDO = str(ranking.iloc[0].modelo)
ESTRATEGIA_ELEGIDA = str(ranking.iloc[0].estrategia)

SELECCION = {
    "modelo": MODELO_ELEGIDO,
    "estrategia": ESTRATEGIA_ELEGIDA,
    "ultima_semana_objetivo_seleccion": int(cv.semana.max()),
    "criterio": "0.60 WAPE SKU + 0.25 WAPE Familia + 0.15 WAPE Planta",
    "puntaje_desarrollo": float(ranking.iloc[0].puntaje),
    "configuracion": CFG,
}
assert SELECCION["ultima_semana_objetivo_seleccion"] == 136

guardar_json(SELECCION, "seleccion_cerrada_antes_prueba.json")
FIRMA_SELECCION = hashlib.sha256(
    json.dumps(SELECCION, sort_keys=True).encode()
).hexdigest()
print("Selección cerrada:", MODELO_ELEGIDO, ESTRATEGIA_ELEGIDA)
print("Huella SHA-256:", FIRMA_SELECCION)


In [ ]:
cv_elegido = cv[
    (cv.modelo == MODELO_ELEGIDO)
    & (cv.estrategia == ESTRATEGIA_ELEGIDA)
].copy()

# Para el mismo modelo, comparar el efecto de cada reconciliación.
tabla(
    resumir(cv[cv.modelo == MODELO_ELEGIDO], ["estrategia", "nivel"]),
    "06_jerarquias_mismo_modelo", max_filas=9,
)

estabilidad = resumir(cv_elegido, ["origen", "nivel"])
tabla(estabilidad, "06_estabilidad_por_origen", mostrar=False)


## 9. Calibración de incertidumbre en desarrollo

La dispersión de errores **fuera de muestra** por nodo y horizonte es la base de las bandas aproximadas del pronóstico. `sigma_error` y `sesgo_calibracion` salen solo del desarrollo y nunca se recalibran mirando la prueba. Las bandas normal-aproximadas deben validarse: pueden **subcubrir** el nivel esperado.

In [ ]:
# La calibración utiliza ÚNICAMENTE errores del desarrollo.
calibracion = (
    cv_elegido[cv_elegido.evaluable]
    .groupby(["nodo", "nivel", "h"])
    .e.agg(
        sesgo_calibracion="mean",
        sigma_error="std",
        n_calibracion="size",
    )
    .reset_index()
)

assert len(calibracion) == len(NODOS) * CFG["h"]
assert calibracion.n_calibracion.min() >= 8
assert np.isfinite(calibracion.sigma_error).all()
tabla(calibracion, "07_sigma_y_sesgo_por_nodo_h", mostrar=False)

FIRMA_CALIBRACION = hashlib.sha256(
    calibracion.to_csv(index=False).encode()
).hexdigest()


In [ ]:
def con_intervalos(pronosticos, calibracion):
    """Añade bandas aproximadas del 80 % y 95 % basadas en errores de desarrollo."""
    datos = pronosticos.merge(
        calibracion, on=["nodo", "nivel", "h"],
        how="left", validate="many_to_one"
    )
    assert datos.sigma_error.notna().all()

    for nivel in [80, 95]:
        z = norm.ppf((1 + nivel / 100) / 2)
        centro = datos.yhat + datos.sesgo_calibracion
        datos[f"lo{nivel}"] = np.maximum(0, centro - z * datos.sigma_error)
        datos[f"hi{nivel}"] = np.maximum(0, centro + z * datos.sigma_error)

    assert (datos.lo95 <= datos.lo80 + 1e-8).all()
    assert (datos.hi95 >= datos.hi80 - 1e-8).all()
    return datos


## 10. Prueba externa: semanas 137 a 156

Recién ahora evaluamos la **receta ya elegida** y el benchmark ingenuo estacional. La prueba no puede cambiar el ganador. Para bandas reportamos **cobertura**, **ancho** y una puntuación que penaliza estar fuera del intervalo. Con ventanas superpuestas, estas métricas describen esta prueba y no garantizan cobertura en el futuro.

In [ ]:
# El modelo y la estrategia fueron elegidos ANTES de mirar esta prueba.
modelos_test = list(dict.fromkeys([MODELO_ELEGIDO, "Estacional"]))
estrategias_test = list(dict.fromkeys([ESTRATEGIA_ELEGIDA, "BU"]))
prueba, audit_test = evaluar_ventanas(
    ventas, CFG["origenes_prueba"],
    modelos_test, estrategias_test, "prueba",
)

test = prueba[
    (prueba.modelo == MODELO_ELEGIDO)
    & (prueba.estrategia == ESTRATEGIA_ELEGIDA)
].copy()
base_test = prueba[
    (prueba.modelo == "Estacional")
    & (prueba.estrategia == "BU")
].copy()

test = con_intervalos(test, calibracion)
tabla(test, "08_predicciones_prueba_con_intervalos", mostrar=False)
met_test = tabla(resumir(test, ["nivel"]), "08_metricas_prueba_por_nivel")
tabla(resumir(base_test, ["nivel"]), "08_benchmark_prueba_por_nivel")
tabla(resumir(test, ["nivel", "h"]),
      "08_metricas_prueba_por_horizonte", max_filas=39)
tabla(resumir(test, ["nodo", "nivel"]), "08_metricas_prueba_por_nodo")

# Evaluación secundaria solo en semanas con ventas realmente observadas.
observados = test[(test.nivel == "SKU") & test.sin_censura_ni_unidad]
tabla(resumir(observados, ["nodo"]),
      "08_desempeno_sin_censura_ni_error_unidad")


In [16]:
def evaluar_intervalos(datos, grupos):
    """Cobertura real y amplitud de cada nivel de banda en datos evaluables."""
    filas = []
    for clave, grupo in datos[datos.evaluable].groupby(grupos, sort=False):
        if not isinstance(clave, tuple):
            clave = (clave,)

        for nivel in [80, 95]:
            inferior = grupo[f"lo{nivel}"]
            superior = grupo[f"hi{nivel}"]
            alpha = 1 - nivel / 100

            proporcion_dentro = (
                (grupo.y >= inferior) & (grupo.y <= superior)
            ).mean()
            puntuacion = (
                (superior - inferior)
                + (2 / alpha) * (inferior - grupo.y).clip(lower=0)
                + (2 / alpha) * (grupo.y - superior).clip(lower=0)
            )
            filas.append({
                **dict(zip(grupos, clave)),
                "intervalo_pct": nivel,
                "n": len(grupo),
                "cobertura_pct": 100 * proporcion_dentro,
                "ancho_medio_cajas": (superior - inferior).mean(),
                "interval_score": puntuacion.mean(),
            })
    return pd.DataFrame(filas)


In [17]:
cobertura = tabla(evaluar_intervalos(test, ['nivel']), '08_cobertura_prueba_por_nivel')

tabla(evaluar_intervalos(test, ['nivel', 'h']), '08_cobertura_prueba_por_horizonte', mostrar=False)

tabla(evaluar_intervalos(test, ['nodo']), '08_cobertura_prueba_por_nodo', max_filas=26)


NameError: name 'tabla' is not defined

In [ ]:
# Comprobar que la prueba no alteró ni selección ni calibración.
assert FIRMA_SELECCION == hashlib.sha256(
    json.dumps(SELECCION, sort_keys=True).encode()
).hexdigest()
assert FIRMA_CALIBRACION == hashlib.sha256(
    calibracion.to_csv(index=False).encode()
).hexdigest()
assert test.semana.min() > SELECCION["ultima_semana_objetivo_seleccion"]

# Una visualización muestra precisión y otra la cobertura de incertidumbre.
fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
for nivel in ["SKU", "Familia", "Planta"]:
    errores_nivel = resumir(test, ["nivel", "h"])
    errores_nivel = errores_nivel[errores_nivel.nivel == nivel]
    ejes[0].plot(
        errores_nivel.h, errores_nivel.WAPE_pct, marker="o", label=nivel
    )

    cobertura_nivel = evaluar_intervalos(test, ["nivel", "h"])
    cobertura_nivel = cobertura_nivel[
        (cobertura_nivel.nivel == nivel)
        & (cobertura_nivel.intervalo_pct == 95)
    ]
    ejes[1].plot(
        cobertura_nivel.h, cobertura_nivel.cobertura_pct,
        marker=".", label=nivel,
    )

ejes[0].set(title="Error externo por horizonte",
            xlabel="h (semanas)", ylabel="WAPE (%)")
ejes[1].axhline(95, color="black", ls="--", label="95% nominal")
ejes[1].set(title="Cobertura externa de bandas 95%",
            xlabel="h (semanas)", ylabel="Cobertura (%)", ylim=(0, 103))
for ax in ejes:
    ax.legend(fontsize=8)
figura(fig, "08_error_y_cobertura_externos")


## 11. Controles reproducibles contra filtración de futuro (*data leakage*)

No basta con escribir «sin leakage»: perturbamos de manera artificial las semanas futuras y exigimos que **el pasado preparado y sus pronósticos queden idénticos**. También revisamos que el ingenuo estacional y la escala MASE se puedan reconstruir a mano con información de entrenamiento.

In [ ]:
comprobaciones = []


def comprobar(nombre, condicion):
    assert bool(condicion), nombre
    comprobaciones.append({"comprobacion": nombre, "resultado": "OK"})


# Perturbar datos futuros deliberadamente NO debe cambiar pronósticos pasados.
corte = 110
alteradas = ventas.copy()
futuro = alteradas.semana > corte
alteradas.loc[futuro, "ventas_cajas"] = (
    alteradas.loc[futuro, "ventas_cajas"].fillna(10) * 1000 + 999
)
alteradas.loc[futuro, "dias_sin_stock"] = 0

original_past, _ = preparar(ventas[ventas.semana <= corte])
alterado_past, _ = preparar(alteradas[alteradas.semana <= corte])
comprobar("Limpieza no cambia al perturbar el futuro",
          original_past.equals(alterado_past))

pred_original, hist_original = predecir_origen(
    ventas, corte, [MODELO_ELEGIDO], ["BU", "TD", "MO"]
)
pred_alterado, hist_alterado = predecir_origen(
    alteradas, corte, [MODELO_ELEGIDO], ["BU", "TD", "MO"]
)
comprobar("Pronóstico y proporciones invariantes al futuro", all(
    np.allclose(pred_original[clave], pred_alterado[clave])
    for clave in pred_original
))

# El prefijo de una limpieza larga coincide con limpiar el prefijo corto.
transformacion_larga, _ = preparar(desarrollo)
comprobar("Transformación causal dentro del prefijo", np.allclose(
    original_past.demanda_estimada,
    transformacion_larga[
        transformacion_larga.semana <= corte
    ].demanda_estimada,
))

auditoria = pd.concat([audit_dev, audit_test], ignore_index=True)
comprobar("Ningún ajuste incluye una semana objetivo",
          (auditoria.max_ajuste < auditoria.min_objetivo).all())
comprobar("20 orígenes de desarrollo y 8 externos",
          len(audit_dev) == 20 and len(audit_test) == 8)
comprobar("Horizontes completos en cada origen", (
    cv.groupby(["modelo", "estrategia", "origen"]).h.nunique() == 13
).all())


In [ ]:
# Ejemplo verificable a mano: ingenuo estacional en la semana 105.
sku = "CIT-L15"
h = 1
entrenamiento_manual, _ = preparar(ventas[ventas.semana <= 104])
historia_manual = jerarquia(entrenamiento_manual)

# Para h=1 desde origen 104, el SNaive repite la semana 53.
pronostico_manual = float(historia_manual.loc[104 + h - 52, sku])
pronostico_codigo = float(cv[
    (cv.modelo == "Estacional")
    & (cv.estrategia == "BU")
    & (cv.origen == 104)
    & (cv.h == h)
    & (cv.nodo == sku)
].yhat.iloc[0])
comprobar("Ingenuo estacional: yhat105 = y53",
          np.isclose(pronostico_manual, pronostico_codigo))

# Denominador de MASE: promedio de errores del naive estacional del entrenamiento.
serie = historia_manual[sku].to_numpy()
escala_manual = np.mean(np.abs(serie[52:] - serie[:-52]))
escala_guardada = cv[
    (cv.origen == 104) & (cv.nodo == sku)
].escala_mase.iloc[0]
comprobar("Denominador MASE calculado solo con entrenamiento",
          np.isclose(escala_manual, escala_guardada))

print(f"Comprobación manual {sku}:")
print(f"  SNaive h=1: {pronostico_manual:.3f} cajas")
print(f"  Escala MASE: {escala_manual:.3f} cajas")


In [ ]:
comprobar("Desarrollo termina en 136 y prueba comienza 137",
          cv.semana.max() == 136 and test.semana.min() == 137)
comprobar("Datos crudos conservan las huellas SHA-256 del original", all(
    hashlib.sha256((CRUDOS / nombre).read_bytes()).hexdigest() == sha
    for nombre, sha in HASH_ESPERADO.items()
))
tabla(pd.DataFrame(comprobaciones), "09_comprobaciones")
tabla(auditoria, "09_auditoria_todas_ventanas", mostrar=False)


## 12. Pronóstico final de las 13 semanas: SKU, familia y planta

Una vez cerrada la prueba, reentrenamos la **misma receta elegida** con las 156 semanas disponibles y estimamos las semanas 157–169. La tabla `10_traspaso_parte_C_pronostico_sigma.csv` es la entrada de la planificación futura. No incorporar merma/sobrellenado aquí: pertenecen a la evidencia de calidad de 1.A.

In [ ]:
# Reentrenar sobre las 156 semanas disponibles, SIN cambiar la receta elegida.
pred_final, hist_final = predecir_origen(
    ventas, 156, [MODELO_ELEGIDO], [ESTRATEGIA_ELEGIDA]
)
pronosticos = pred_final[(MODELO_ELEGIDO, ESTRATEGIA_ELEGIDA)]
pf = pronosticos  # Alias usado después en el análisis de sensibilidad.

fechas = pd.date_range(
    calendario.loc[156] + pd.Timedelta(weeks=1),
    periods=13, freq="W-MON",
)

registros = []
for h in range(1, 14):
    for nodo in NODOS:
        registros.append({
            "nodo": nodo, "nivel": NIVEL[nodo],
            "h": h, "semana": 156 + h,
            "fecha_lunes": fechas[h - 1],
            "yhat": pronosticos.loc[h, nodo],
            "modelo": MODELO_ELEGIDO,
            "estrategia": ESTRATEGIA_ELEGIDA,
        })

final = con_intervalos(pd.DataFrame(registros), calibracion)
final_sku = final[final.nivel == "SKU"].rename(
    columns={"nodo": "sku", "yhat": "pronostico_cajas"}
)
final_sku = final_sku.merge(
    maestro[["sku", "familia", "linea_envasado", "litros_por_caja"]],
    on="sku", validate="many_to_one",
)

tabla(final_sku, "10_pronostico_final_sku_13_semanas", max_filas=18)
tabla(final, "10_pronostico_final_todos_niveles", mostrar=False)


### 12.1. Tabla para planificación y pronóstico de errores

Distinguimos `sigma_i_h_desarrollo` (calibrado antes de prueba) de `sigma_prueba` (solo contraste externo). Para errores acumulados en varios períodos, estimamos directamente la suma de errores, sin asumir independencia semanal.

In [ ]:
# Distinguimos sigma de desarrollo (para decisión) y sigma de prueba (auditoría).
sigma_test = (
    test[(test.nivel == "SKU") & test.evaluable]
    .groupby(["nodo", "h"])
    .e.agg(sigma_prueba="std", sesgo_prueba="mean", n_prueba="size")
    .reset_index()
    .rename(columns={"nodo": "sku"})
)

columnas_entrega = [
    "sku", "familia", "linea_envasado", "h", "semana", "fecha_lunes",
    "pronostico_cajas", "sigma_error", "sesgo_calibracion",
    "n_calibracion", "lo80", "hi80", "lo95", "hi95", "litros_por_caja",
]
traspaso = final_sku[columnas_entrega].merge(
    sigma_test, on=["sku", "h"]
)
traspaso = traspaso.rename(columns={
    "sigma_error": "sigma_i_h_desarrollo",
    "sesgo_calibracion": "sesgo_i_h_desarrollo",
})
traspaso["litros_nominales_pronosticados"] = (
    traspaso.pronostico_cajas * traspaso.litros_por_caja
)
traspaso["origen_sigma"] = "validación desarrollo, pipeline reconciliado"
tabla(traspaso, "10_traspaso_parte_C_pronostico_sigma", mostrar=False)

# Sigma del ERROR ACUMULADO para diferentes lead times L.
acumulados = []
for L in [1, 2, 4, 13]:
    elegibles = cv_elegido[
        (cv_elegido.nivel == "SKU")
        & cv_elegido.evaluable
        & (cv_elegido.h <= L)
    ]
    for (sku, origen), grupo in elegibles.groupby(["nodo", "origen"]):
        if len(grupo) == L:
            acumulados.append({
                "sku": sku, "origen": origen,
                "L": L, "error_acumulado": grupo.e.sum(),
            })

sigma_acum = (
    pd.DataFrame(acumulados)
    .groupby(["sku", "L"])
    .error_acumulado.agg(
        sigma_error_acumulado="std", sesgo_acumulado="mean",
        n_origenes="size",
    )
    .reset_index()
)
tabla(sigma_acum, "10_sigma_error_acumulado_lead_time", mostrar=False)


### 12.2. Auditoría de coherencia y figura final

In [ ]:
comprobar("Pronóstico final tiene 117 filas SKU y 13 semanas",
          len(final_sku) == 117 and final_sku.h.nunique() == 13)
comprobar("Fechas finales coinciden con horizonte del caso",
          fechas[0] == pd.Timestamp("2026-09-28")
          and fechas[-1] == pd.Timestamp("2026-12-21"))

# Reconciliación: comprobar que se cumplen las sumas EN CADA semana futura.
for h, grupo in final.groupby("h"):
    pronostico_h = grupo.set_index("nodo").yhat
    cuadra_planta = np.isclose(pronostico_h.PLANTA, pronostico_h[SKUS].sum())
    cuadra_familias = all(
        np.isclose(
            pronostico_h["FAM_" + familia],
            pronostico_h[[sku for sku in SKUS if FAM[sku] == familia]].sum(),
        ) for familia in FAMILIAS
    )
    comprobar(f"Coherencia puntual final h={h}",
              cuadra_planta and cuadra_familias)

# Datos originales y transformados se guardan por separado.
lim_final, bit_final = preparar(ventas)
tabla(bit_final, "10_bitacora_completa", mostrar=False)
tabla(lim_final, "10_datos_transformados_trazables", mostrar=False)

fig, ejes = plt.subplots(3, 3, figsize=(15, 10))
for ax, sku in zip(ejes.flat, SKUS):
    historia = lim_final[lim_final.sku == sku].tail(52)
    futuro = final_sku[final_sku.sku == sku].sort_values("h")

    ax.plot(historia.fecha_lunes, historia.demanda_estimada,
            color="#64748b", lw=1, label="Historia estimada")
    ax.fill_between(futuro.fecha_lunes, futuro.lo95, futuro.hi95,
                    color="#21768a", alpha=0.14, label="95% aproximado")
    ax.fill_between(futuro.fecha_lunes, futuro.lo80, futuro.hi80,
                    color="#21768a", alpha=0.28, label="80% aproximado")
    ax.plot(futuro.fecha_lunes, futuro.pronostico_cajas,
            color="#075367", lw=2, label="Puntual")
    ax.axvline(fechas[0], color="gray", ls=":", lw=1)
    ax.set_title(sku)
    ax.set_ylabel("Cajas/semana")
    ax.tick_params(axis="x", rotation=30)

ejes.flat[0].legend(fontsize=7)
fig.suptitle(
    f"Próximas 13 semanas · {MODELO_ELEGIDO} / {ESTRATEGIA_ELEGIDA}",
    y=1.02,
)
figura(fig, "10_pronostico_final_9_sku_intervalos")


## 13. Sensibilidad de decisiones no identificadas por los datos

Una cifra única depende de supuestos sobre demanda perdida y errores de unidad. Por eso mantenemos fija la receta y comparamos recuperación del 0 %, 50 % y 100 % más el caso sin corrección de unidad. No son estimaciones de probabilidades reales.

In [ ]:
# Repetimos solo la predicción final; NO volvemos a seleccionar modelos.
escenarios = [
    ("recuperacion_0", 0.0, True),
    ("recuperacion_50", 0.5, True),
    ("base_recuperacion_100", 1.0, True),
    ("sin_corregir_unidad", 1.0, False),
]

sens = []
for nombre, recuperacion, corregir_unidad in escenarios:
    if nombre == "base_recuperacion_100":
        pronostico_escenario = pf
    else:
        proyecciones, _ = predecir_origen(
            ventas, 156, [MODELO_ELEGIDO], [ESTRATEGIA_ELEGIDA],
            recuperacion, corregir_unidad,
        )
        pronostico_escenario = proyecciones[(MODELO_ELEGIDO, ESTRATEGIA_ELEGIDA)]

    for sku in SKUS:
        base = pf[sku].sum()
        total = pronostico_escenario[sku].sum()
        sens.append({
            "escenario": nombre,
            "sku": sku,
            "total_13_semanas_cajas": total,
            "cambio_vs_base_pct": 100 * (total / base - 1) if base else np.nan,
        })

sensibilidad = tabla(pd.DataFrame(sens), "11_sensibilidad_supuestos", max_filas=36)


## 14. Resultados principales

La conclusión se genera a partir de las tablas calculadas, sin escribir números manualmente. Distingue: decisión seleccionada; precisión y sesgo externos; cobertura de bandas; intermitencia; y limitaciones de las hipótesis.

In [18]:
# Todo número del texto final se extrae de tablas calculadas arriba.
avisos = pd.DataFrame(AVISOS)
tabla(avisos, "12_registro_ajustes_y_avisos", mostrar=False)

retornos = (
    avisos.assign(retorno=avisos.usado.eq("RETORNO_ESTACIONAL"))
    .groupby("candidato")
    .agg(
        ajustes=("usado", "size"),
        retornos=("retorno", "sum"),
        recortes_negativos=("recortes_negativos", "sum"),
    )
    .reset_index()
)
tabla(retornos, "12_salud_ajustes")

met_sku = met_test.set_index("nivel").loc["SKU"]
bench_sku = resumir(base_test, ["nivel"]).set_index("nivel").loc["SKU"]
inter = diagnostico[diagnostico.ADI >= 1.32].sku.tolist()
elegido_dev = ranking.iloc[0]
c95 = cobertura[
    (cobertura.nivel == "SKU") & (cobertura.intervalo_pct == 95)
].iloc[0]
c80 = cobertura[
    (cobertura.nivel == "SKU") & (cobertura.intervalo_pct == 80)
].iloc[0]
sumas = final_sku.groupby("sku").pronostico_cajas.sum().sort_values(ascending=False)

# Construcción del resumen a partir de los resultados calculados.
lineas = ["### Resultados del análisis", ""]
lineas.append(
    f"- **Receta elegida solo en desarrollo:** {MODELO_ELEGIDO} + "
    f"{ESTRATEGIA_ELEGIDA}; puntaje ponderado {elegido_dev.puntaje:.2f}."
)

mejora = "mejora" if met_sku.WAPE_pct < bench_sku.WAPE_pct else "no mejora"
lineas.append(
    f"- **Prueba externa SKU:** WAPE {met_sku.WAPE_pct:.2f}%, "
    f"MASE {met_sku.MASE:.3f}, sesgo {met_sku.sesgo_cajas:.2f} cajas por par "
    f"y sesgo relativo {met_sku.sesgo_pct:.2f}%. "
    f"El ingenuo estacional BU obtiene WAPE {bench_sku.WAPE_pct:.2f}%. "
    f"La receta elegida {mejora} esa referencia en esta prueba; "
    "no se cambia de ganador retrospectivamente."
)

advertencia_95 = (
    "La banda95 subcubre: debe advertirse al planificar."
    if c95.cobertura_pct < 95 else
    "La banda95 alcanza o supera el nominal agregado en esta muestra; "
    "revisar por SKU y horizonte."
)
advertencia_80 = (
    "es conservadora en esta muestra y su anchura puede elevar el inventario."
    if c80.cobertura_pct > 85 else
    "revisar anchura y cobertura por SKU antes de usar las bandas para stock."
)
lineas.append(
    f"- **Bandas externas SKU:** cobertura80 = {c80.cobertura_pct:.2f}% y "
    f"cobertura95 = {c95.cobertura_pct:.2f}%. {advertencia_95} "
    f"La cobertura80 difiere del nominal en {c80.cobertura_pct - 80:+.2f} puntos: "
    f"{advertencia_80}"
)
lineas.append(
    f"- **Intermitencia en desarrollo:** {', '.join(inter) or 'ningún SKU con ADI≥1,32'}. "
    "Los ceros se conservan y se contrasta SBA sin imponer su selección."
)
lineas.append(
    f"- **Proyección final:** {final_sku.pronostico_cajas.sum():,.1f} cajas "
    f"en 13 semanas; mayor volumen acumulado: {sumas.index[0]} "
    f"({sumas.iloc[0]:,.1f} cajas). "
    "La suma por familia y planta coincide exactamente con los SKU."
)
lineas.extend(["",
    "**Interpretación y límites:** el objetivo reconstruido no es demanda observada; "
    "los días de quiebre no revelan qué ocurrió dentro del día. La promoción no "
    "es aleatoria y su asociación con ventas no identifica elasticidad causal. "
    "La fase estacional familiar y los errores por nivel permiten explicar "
    "los límites del reparto fijo TD. Las bandas son aproximadas y los errores "
    "de ventanas solapadas están correlacionados. El ranking no es evidencia "
    "de dominancia universal; hay solo tres ciclos anuales.",
    "",
    "**Entrada a E2:** usar `10_traspaso_parte_C_pronostico_sigma.csv`, "
    "incluyendo fecha, h, pronóstico, σ, sesgo y origen de calibración. "
    "Integrar merma por línea y sobrellenado desde 1.A; no derivarlos de ventas ni inventarlos.",
])

salida = "\n".join(lineas) + "\n"
display(Markdown(salida))
(SAL / "conclusiones_generadas.md").write_text(salida, encoding="utf-8")

guardar_json({
    "seleccion": SELECCION,
    "firma": FIRMA_SELECCION,
    "metricas_prueba": met_test.to_dict("records"),
    "cobertura_prueba": cobertura.to_dict("records"),
    "pronostico_total_cajas": float(final_sku.pronostico_cajas.sum()),
}, "resumen_ejecutivo.json")


NameError: name 'AVISOS' is not defined

### Interpretación de los resultados

- **¿Por qué no pronosticar ventas sin revisar stock?** Porque las ventas se censuran cuando no está disponible el producto; la demanda estimada depende de un supuesto explícito.
- **¿Por qué validación por origen rodante?** Porque cada modelo debe predecir con el conjunto de información disponible **en ese lunes**, no con todo el período observado.
- **¿Por qué no escoger el modelo de menor error solo en la prueba?** Porque se convertiría la evaluación final en entrenamiento encubierto.
- **¿Qué significan las bandas?** Una estimación aproximada de incertidumbre del error; hay que contrastar su cobertura real.


**Fuentes del curso utilizadas:** `CaseStudy/Enunciado.pdf`, `CaseStudy/Rubricas/Entrega_1.pdf`, `Scripts/2_Pronostico/01`, `02`, `04` y `05`; `Laboratorio/Pautas/Sesion_3` y diapositivas `3_Pronostico.pdf`.

**Supuestos metodológicos:** los pesos de selección, el ajuste por quiebres de stock, los umbrales y las bandas normales aproximadas requieren justificación y análisis de sensibilidad.

## 15. Exportaciones y registro del entorno

Guarda versiones, hashes, configuración, comprobaciones y resultados en `salidas_avance1b/`. El ZIP resultante se descarga desde el panel de Archivos de Colab. Nada se sube automáticamente al GitHub del equipo.

In [ ]:
# Archivo de resultados y evidencia de reproducibilidad.
tabla(pd.DataFrame(comprobaciones), "09_comprobaciones", mostrar=False)

entorno = {
    "python": sys.version,
    "plataforma": platform.platform(),
    "paquetes": {paquete: im.version(paquete) for paquete in VERSIONES},
    "semilla": SEED,
    "commit_docente": CFG["commit_fuentes"],
    "firma_seleccion": FIRMA_SELECCION,
    "firma_calibracion": FIRMA_CALIBRACION,
}
guardar_json(entorno, "entorno_ejecucion.json")

(SAL / "requirements.txt").write_text(
    "\n".join(f"{p}=={v}" for p, v in VERSIONES.items()) + "\n",
    encoding="utf-8",
)
(SAL / "LEEME.txt").write_text(
    "Avance 1.B - Pronóstico de demanda. Ejecutar todas las celdas en orden.\n"
    "El archivo 10_traspaso_parte_C_pronostico_sigma.csv es la salida para 1.C.\n"
    "Integrar calidad desde 1.A; crudos inmutables y correcciones trazables.\n"
    "Seleccion cerrada con objetivos hasta 136; prueba externa 137-156.\n",
    encoding="utf-8",
)

archivo_zip = Path(shutil.make_archive(
    str(SAL.parent / "Resultados_Avance1B"), "zip", SAL
))
print(f"Ejecución completa: {len(comprobaciones)} controles; "
      f"{len(final_sku)} pronósticos de SKU.")
display(FileLink(str(archivo_zip.relative_to(Path.cwd()))))
